# 01: Data Preprocessing and Validation

**Objective:** This notebook is the first analytical step in the pipeline. It is responsible for:
1.  **Loading all raw data** from the `/data/raw/` directory using the resilient `data_loader` utility.
2.  **Performing transparent deduplication** of studies to create a master, unique list of records.
3.  **Merging and cleaning** the various datasets into analysis-ready dataframes.
4.  **Saving the processed dataframes** to the `/data/processed/` directory for use in downstream analysis notebooks.

**Guiding Principles:**
- **Extreme Robustness:** The notebook must complete without error even if the data directory is empty.
- **Full Auditability:** All major decisions, especially deduplication, must be explicitly logged.

In [1]:
# === 1. SETUP: IMPORTS, PATHS, AND LOGGING ===
import os
import sys
import pandas as pd
import logging

# --- Path Configuration ---
# Add the project root to the Python path to allow importing from 'utils'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Define key directories
data_dir = os.path.join(project_root, 'data')
processed_dir = os.path.join(project_root, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)

# --- Logging ---
# Get the existing logger instance initialized by the master orchestrator
logger = logging.getLogger("amr_ssi_pipeline")
logger.info("--- Starting Notebook 01: Data Preprocessing ---")

# --- Import Custom Utilities ---
from utils.data_loader import load_all_data
from utils.audit_utils import log_exclusion

print("Setup complete. Data directory:", data_dir)

Setup complete. Data directory: c:\Users\Marion Korir\code\amr-ssi-python-analysis\data


In [2]:
# === 2. LOAD ALL RAW DATA ===
# Use the resilient data loader to ingest all files from the data directory.
# This returns a dictionary of DataFrames.
logger.info(f"Loading all raw data from: {data_dir}")
raw_data_dict = load_all_data(data_dir)

# Check if any data was loaded
if not raw_data_dict:
    logger.warning("No data files were loaded. The pipeline will continue but no processing will occur.")
    # This allows the notebook to complete successfully even with no input.
else:
    logger.info(f"Successfully loaded {len(raw_data_dict)} dataframes: {list(raw_data_dict.keys())}")
    # Display the first few rows of each loaded dataframe for a quick check
    for name, df in raw_data_dict.items():
        print(f"--- First 5 rows of {name} ---")
        display(df.head())

No data files were successfully loaded from c:\Users\Marion Korir\code\amr-ssi-python-analysis\data.


No data files were loaded. The pipeline will continue but no processing will occur.


### 3. Deduplication and Merging

This section is a placeholder for the deduplication and merging logic. The specific implementation will depend on the structure of the input files. A common approach is:

1.  **Identify a primary dataframe** (e.g., from `ssi_rollup.csv` or a similar file) that contains the core list of studies.
2.  **Define deduplication criteria** (e.g., a combination of author, year, and title).
3.  **Log every duplicate** found and the criteria for its removal to a dedicated log file (`logs/deduplication_log.md`).
4.  **Merge** the cleaned, unique primary dataframe with other dataframes (e.g., `amr_proportions_long.csv`, `costs_long.csv`) using a common study identifier.

In [3]:
# === 3.1. DEDUPLICATION LOGIC (PLACEHOLDER) ===

# This is a placeholder for the actual deduplication logic.
# We will first check if the required 'ssi_rollup' dataframe exists.

if 'ssi_rollup' in raw_data_dict:
    logger.info("Found 'ssi_rollup' dataframe. Proceeding with placeholder deduplication.")
    
    # Assume 'ssi_rollup' is the primary source for the list of studies
    master_df = raw_data_dict['ssi_rollup'].copy()
    
    # --- Placeholder Deduplication ---
    # In a real scenario, this would involve checking for similar author/year/title.
    # For now, we will use pandas' built-in duplicate detection on a key column.
    # Let's assume a column named 'study_id' is the unique identifier.
    
    if 'study_id' in master_df.columns:
        initial_count = len(master_df)
        duplicates = master_df[master_df.duplicated(subset=['study_id'], keep=False)]
        
        if not duplicates.empty:
            logger.warning(f"Found {len(duplicates)} duplicate entries based on 'study_id'.")
            
            # Log the details of duplicates to a markdown file for auditability
            dedup_log_path = os.path.join(project_root, 'logs', 'deduplication_log.md')
            with open(dedup_log_path, 'w') as f:
                f.write("# Deduplication Log\n\n")
                f.write(f"Timestamp: {pd.Timestamp.now()}\n\n")
                f.write(f"Found {len(duplicates)} records with duplicate 'study_id's. The first instance of each was kept.\n\n")
                f.write("## Duplicate Records Identified:\n\n")
                f.write(duplicates.to_markdown(index=False))

            # Remove duplicates, keeping the first instance
            master_df.drop_duplicates(subset=['study_id'], keep='first', inplace=True)
            final_count = len(master_df)
            logger.info(f"Removed {initial_count - final_count} duplicates. Final count: {final_count}.")
            
        else:
            logger.info("No duplicates found based on 'study_id'.")
            
    else:
        logger.warning("'study_id' column not found in 'ssi_rollup'. Skipping deduplication.")
        # In a real scenario, you might try other columns or combinations.

    # --- Placeholder Merging ---
    # Here you would merge the master_df with other relevant dataframes.
    # For example, merging with AMR data:
    if 'amr_proportions_long' in raw_data_dict:
        amr_df = raw_data_dict['amr_proportions_long']
        if 'study_id' in amr_df.columns:
            # Merge, keeping only records that exist in the cleaned master list
            merged_df = pd.merge(master_df, amr_df, on='study_id', how='left')
            logger.info("Placeholder: Merged 'master_df' with 'amr_proportions_long'.")
            # For this example, we'll just use the master_df as the final processed df
            processed_df = master_df
        else:
            logger.warning("Cannot merge AMR data: 'study_id' not found in 'amr_proportions_long'.")
            processed_df = master_df
    else:
        processed_df = master_df

else:
    logger.warning("'ssi_rollup' dataframe not found. Cannot perform deduplication or create a master dataframe.")
    processed_df = pd.DataFrame() # Create an empty dataframe to allow the notebook to run


# Display the final processed dataframe if it's not empty
if not processed_df.empty:
    print("\n--- Final Processed DataFrame (Sample) ---")
    display(processed_df.head())

'ssi_rollup' dataframe not found. Cannot perform deduplication or create a master dataframe.


In [4]:
# === 4. SAVE PROCESSED DATA ===
# Save the final, cleaned, and merged dataframe to the processed data directory.
# This file will be the input for all subsequent analysis notebooks.

if not processed_df.empty:
    output_path = os.path.join(processed_dir, 'master_analysis_data.csv')
    try:
        processed_df.to_csv(output_path, index=False)
        logger.info(f"Successfully saved processed data to: {output_path}")
        print(f"\nProcessed data saved to: {output_path}")
    except Exception as e:
        logger.error(f"Failed to save processed data. Error: {e}", exc_info=True)
else:
    logger.warning("Processed dataframe is empty. Nothing to save.")

Processed dataframe is empty. Nothing to save.


---
## End of Notebook 01
---